<a href="https://colab.research.google.com/github/PriyanshuChaudhary00/CNN_Using_Transfer_learning/blob/main/Notebooks/experimental.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1️⃣ Introduction

This project performs binary image classification (Cat vs Dog) using Transfer Learning with the VGG16 pretrained model in PyTorch.

Instead of training a CNN from scratch, the pretrained convolutional layers of VGG16 are reused while replacing the classifier for the binary classification task.

* Key Features


* Transfer Learning using VGG16


* PyTorch implementation


* Binary Classification


* Model Checkpoint Saving


* Confusion Matrix


* Classification Report


* Prediction Script

2️⃣ Problem Statement

The goal of this project is to classify an input image as either:

🐱 Cat

🐶 Dog

using Transfer Learning to improve training efficiency and accuracy.

4️⃣ Project Architecture

                    Input Images
                          │
                          ▼
                Image Preprocessing
     Resize → Tensor → Normalize
                          │
                          ▼
                    DataLoader
                          │
                          ▼
          Pretrained VGG16 Feature Extractor
             (Frozen Convolution Layers)
                          │
                          ▼
             Custom Fully Connected Layer
                          │
                          ▼
                  Binary Classification
                    Cat        Dog
                          │
                          ▼
                 BCEWithLogitsLoss
                          │
                          ▼
                     Adam Optimizer
                          │
                          ▼
                   Trained Model (.pth)
                          │
                          ▼
                 Prediction & Evaluation

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset , DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from sklearn.metrics import classification_report , confusion_matrix
import matplotlib.pyplot as plt

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'dog-and-cat-classification-dataset' dataset.
Path to dataset files: /kaggle/input/dog-and-cat-classification-dataset


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


In [ ]:
transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

In [ ]:
dataset = ImageFolder(f"{path}/PetImages" , transform=transform)

In [ ]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = random_split(
    dataset,
    [train_size, test_size]
)


In [ ]:
train_loader = DataLoader(dataset=train_dataset , batch_size = 32 , shuffle=True)
test_loader = DataLoader(dataset=test_dataset , batch_size = 32 , shuffle=True)

Importing Model

In [ ]:
import torchvision.models as models
vgg16 = models.vgg16(pretrained = True)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
vgg16.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

In [ ]:
for params in vgg16.features.parameters():
  params.requires_grad = False

<div align="center">

<table>
<tr>
<td align="center" style="border:2px solid #4F81BD;padding:12px;border-radius:8px;">
<b>🖼 Input Image</b>
</td>
</tr>
<tr><td align="center">⬇️</td></tr>

<tr>
<td align="center" style="border:2px solid #4F81BD;padding:12px;border-radius:8px;">
<b>🛠 Image Preprocessing</b><br>
Resize (224×224)<br>
Normalize<br>
ToTensor
</td>
</tr>

<tr><td align="center">⬇️</td></tr>

<tr>
<td align="center" style="border:2px solid #4F81BD;padding:12px;border-radius:8px;">
<b>🧠 Pretrained VGG16</b><br>
Frozen Feature Extractor
</td>
</tr>

<tr><td align="center">⬇️</td></tr>

<tr>
<td align="center" style="border:2px solid #4F81BD;padding:12px;border-radius:8px;">
<b>🎯 Custom Classifier</b><br>
Linear → Dropout → Linear
</td>
</tr>

<tr><td align="center">⬇️</td></tr>

<tr>
<td align="center" style="border:2px solid #4F81BD;padding:12px;border-radius:8px;">
<b>✅ Binary Output</b><br>
Cat 🐱 / Dog 🐶
</td>
</tr>

</table>

</div>

In [ ]:
vgg16.classifier = nn.Sequential(
    nn.Linear(25088 , 1024),
    nn.ReLU(),
    nn.Dropout(0.5),

    nn.Linear(1024 , 512),
    nn.ReLU(),
    nn.Dropout(0.5),

    nn.Linear(512 , 32),
    nn.ReLU(),
    nn.Linear(32 , 1)
)

In [ ]:
vgg16

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [ ]:
vgg16 = vgg16.to(device)

In [ ]:
learning_rate = 0.0001
epochs = 2

In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(vgg16.classifier.parameters() , lr=learning_rate , weight_decay=1e-4)

In [ ]:
train_losses = []
# train_losses.append(epoch_loss)

In [ ]:
for epoch in range(epochs):
  print("start")
  total_loss_in_epoch = 0
  for input , target in test_loader:
    optimizer.zero_grad()
    input , target = input.to(device) , target.unsqueeze(1).float().to(device)

    y_pred = vgg16(input)
    loss = loss_fn(y_pred , target)

    loss.backward()
    optimizer.step()
    total_loss_in_epoch += loss.item()
  total_loss_in_epoch /= len(train_loader)
  print(f"Epoch: {epoch+1}, Loss: {total_loss_in_epoch}")

In [ ]:
vgg16.eval()
correct = 0
total = 0
y_pred = []
y_true = []
with torch.no_grad():
  for image , target in test_loader():
    image , target = image.to(device) , target.float().to(device)

    logit = vgg16(image)
    pred = torch.sigmoid(logit)
    y_hat = (pred > 0.5).float()

    correct += (y_hat == target.unsqueeze(1)).sum().item()
    total += target.size(0)

    y_pred = y_pred.extend(y_hat.cpu().numpy())
    y_true = y_true.extend(target.cpu().numpy())

accuracy = (correct / total)*100
print(f"Model accuracy is : {accuracy}")

In [ ]:
print(classification_report(
    y_true=y_true,
    y_pred=y_pred,
    target_names=["cat" , "dog"]
))

In [ ]:
cm = confusion_matrix(y_true , y_pred)
print(cm)

plt.figure(figsize=(5,5))

plt.imshow(cm, cmap="Blues")

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.xticks([0,1],["Cat","Dog"])
plt.yticks([0,1],["Cat","Dog"])

for i in range(2):
    for j in range(2):
        plt.text(j,i,str(cm[i,j]),
                 ha='center',
                 va='center',
                 color='black')

plt.colorbar()
plt.show()
plt.savefig("results/confusion_matrix.png")


In [ ]:
plt.figure(figsize=(8,5))

plt.plot(train_losses)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")

plt.grid()

plt.show()
plt.savefig("results/loss_curve.png")

In [ ]:
vgg16.eval()

images,labels = next(iter(test_loader))

images = images.to(device)

with torch.no_grad():

    outputs = vgg16(images)

    probs = torch.sigmoid(outputs)

    preds = (probs>0.5).float()

In [ ]:
torch.save(vgg16.state_dict(), "cat_vs_dog_model.pth")